In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.cross_decomposition import PLSRegression

In [2]:
z_score_df = pd.read_pickle('matrix_Lisbon_Coimbra_threshold.pkl')

In [3]:
import pickle

with open('list_groups_LC.pkl', 'rb') as f:
    list_groups = pickle.load(f)

print(list_groups)

['MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-CT', 'MCI-AD',

In [4]:
list_groups = pd.Series(list_groups)

In [5]:
def calculate_vip(pls, X):
    
    t = pls.x_scores_
    w = pls.x_weights_
    q = pls.y_loadings_
    
    p, h = w.shape
    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    
    Wnorm = (w ** 2) / np.sum(w ** 2, axis=0)
    
    vip = np.sqrt(p * (Wnorm @ s) / np.sum(s))
    
    return vip.flatten()

In [6]:
def stratified_bootstrap(X, y):
    """
    Creates a bootstrap sample preserving the original class proportions.
    """
    ad_idx = np.where(y == "MCI-AD")[0]
    ct_idx = np.where(y == "MCI-CT")[0]

    ad_sample = np.random.choice(ad_idx, size=len(ad_idx), replace=True)
    ct_sample = np.random.choice(ct_idx, size=len(ct_idx), replace=True)

    indices = np.concatenate([ad_sample, ct_sample])
    np.random.shuffle(indices)

    return X.iloc[indices], y.iloc[indices]


# Settings
n_iterations = 100
feature_names = z_score_df.columns

importance_matrix = np.zeros((n_iterations, len(feature_names)))
ranking_matrix = np.zeros((n_iterations, len(feature_names)))

vip_matrix = np.zeros((n_iterations, len(feature_names)))
vip_ranking_matrix = np.zeros((n_iterations, len(feature_names)))
vip_selection_matrix = np.zeros((n_iterations, len(feature_names)))

le = LabelEncoder()
le.fit(list_groups)


# Main loop
for i in range(n_iterations):

    X_boot, y_boot = stratified_bootstrap(z_score_df, list_groups)

    # =====================
    # RANDOM FOREST 
    # =====================

    rf = RandomForestClassifier(
        n_estimators=1000,
        class_weight="balanced",
        n_jobs=-1,
        random_state=i
    )

    rf.fit(X_boot, y_boot)

    # importances
    importances = rf.feature_importances_
    importance_matrix[i] = importances

    # ranking (1 = migliore)
    ranks = (-importances).argsort().argsort() + 1
    ranking_matrix[i] = ranks

    # =====================
    # PLS-DA 
    # =====================

    # encode y
    y_encoded = le.transform(y_boot)

    pls = PLSRegression(n_components=2)
    pls.fit(X_boot, y_encoded)

    # VIP
    vip_scores = calculate_vip(pls, X_boot)
    vip_matrix[i] = vip_scores

    # ranking VIP (1 = migliore)
    vip_ranks = (-vip_scores).argsort().argsort() + 1
    vip_ranking_matrix[i] = vip_ranks
    vip_selection_matrix[i] = (vip_scores > 1).astype(int)


# Aggregazione risultati
mean_importance = importance_matrix.mean(axis=0)
mean_rank = ranking_matrix.mean(axis=0)
std_rank = ranking_matrix.std(axis=0)

mean_vip = vip_matrix.mean(axis=0)
mean_vip_rank = vip_ranking_matrix.mean(axis=0)
std_vip_rank = vip_ranking_matrix.std(axis=0)
vip_freq = vip_selection_matrix.mean(axis=0)


# DataFrame finale
importance_df = pd.DataFrame({
    "protein": feature_names,
    "importance": mean_importance,
    "mean_rank": mean_rank,
    "rank_std": std_rank
}).sort_values(by="importance", ascending=False)

vip_df = pd.DataFrame({
    "protein": feature_names,
    "VIP": mean_vip,
    "mean_rank_vip": mean_vip_rank,
    "std_rank_vip": std_vip_rank,
    "vip_freq": vip_freq
}).sort_values(by="VIP", ascending=False)

In [7]:
vip_df

,protein,VIP,mean_rank_vip,std_rank_vip,vip_freq
110,P14618,2.596074,1.09,0.319218,1.0
55,P02766,2.205132,3.99,2.170230,1.0
196,Q9UBX5,2.138411,5.03,2.479738,1.0
168,Q16270,2.127568,5.53,3.251015,1.0
158,Q13449,2.115915,5.60,3.438023,1.0
...,...,...,...,...,...
150,P98160,0.369920,175.06,21.648473,0.0
151,Q02246,0.368392,174.00,26.289922,0.0
97,P09972,0.360619,173.45,32.453159,0.0
105,P12259,0.349820,174.39,30.077864,0.0


In [8]:
importance_df

,protein,importance,mean_rank,rank_std
110,P14618,0.077533,1.08,0.365513
55,P02766,0.042569,3.91,2.025315
158,Q13449,0.039108,5.26,4.107603
168,Q16270,0.038002,5.10,2.823119
127,P25311,0.035223,5.63,3.025409
...,...,...,...,...
19,P00751,0.000842,178.60,20.496341
30,P01599,0.000835,179.63,26.004867
21,P01009,0.000834,179.09,22.138245
95,P08697,0.000755,183.15,24.284306


In [9]:
final_df = importance_df.merge(vip_df, on="protein")
final_df["rank_diff"] = abs(final_df["mean_rank"] - final_df["mean_rank_vip"])
final_df["mean_rank_combined"] = (
    final_df["mean_rank"] + final_df["mean_rank_vip"]
) / 2
final_df["std_rf"] = final_df["rank_std"]
final_df["std_pls"] = final_df["std_rank_vip"]
final_df = final_df[[
    "protein",
    "mean_rank_combined",
    "std_rf",
    "std_pls",
    "vip_freq",
    "rank_diff"
]]
final_df = final_df.sort_values(by="mean_rank_combined")

In [10]:
final_df.head(20)

,protein,mean_rank_combined,std_rf,std_pls,vip_freq,rank_diff
0,P14618,1.085,0.365513,0.319218,1.00,0.01
1,P02766,3.950,2.025315,2.170230,1.00,0.08
3,Q16270,5.315,2.823119,3.251015,1.00,0.43
2,Q13449,5.430,4.107603,3.438023,1.00,0.34
4,P25311,5.730,3.025409,3.215758,1.00,0.20
6,Q9UBX5,6.585,3.784230,2.479738,1.00,3.11
5,Q14118,7.060,3.324620,2.944469,1.00,0.14
8,P02751,9.155,4.831346,3.021854,1.00,1.87
7,P61916,10.220,4.212778,4.165801,1.00,0.94
9,Q9NQ79,10.610,6.149951,4.513657,1.00,0.96
